# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of Analysis + Time Window

**One row means**
One row represents the daily search and analytics performance of one content item (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).

**Table used**
I use the `fact_content_daily_performance` warehouse table, specifically the `month=2026-03` partition for this assignment.

**Time window**
The analysis uses the March 2026 partition (`month = 2026-03`), which is a mid-panel month recommended for feature engineering and verification.

**Prediction / Ranking**
My lane is **Content Refresh Opportunity Scoring**. The objective is to rank content pages according to their refresh priority using historical search and engagement metrics.

**Deliberately excluded**
I exclude rows where `gsc_data_available` is FALSE because Search Console metrics are unavailable for those dates. I also avoid using the final month (June 2026) to reduce the risk of data leakage.

In [4]:
import pandas as pd

march_df = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03"
)

print(march_df.shape)
march_df.head()
print(march_df["report_date"].dtype)

print("Total Rows:", len(march_df))
print("Start Date:", march_df["report_date"].min())
print("End Date:", march_df["report_date"].max())

print("\nUnique Month(s):")
print(sorted(march_df["report_date"].apply(lambda x: x.strftime("%Y-%m")).unique()))

(9841378, 30)
object
Total Rows: 9841378
Start Date: 2026-03-01
End Date: 2026-03-31

Unique Month(s):
['2026-03']


## 2. Fields: Feature / Label / Context / Excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

These are historical metrics available before making the refresh decision.

### Label / Proxy
The label will be created later as a refresh opportunity score (or future performance change). It is intentionally **not used as a feature**.

### Context
- report_date
- client_hash_id
- content_hash_id

These identify records and are used for grouping or joining only.

### Excluded
- gsc_data_available → Used only to filter valid Search Console records.
- ga4_data_available → Availability flag, not predictive.
- client_has_gsc → Infrastructure flag, not page performance.
- client_has_ga4 → Infrastructure flag, not page performance.
- gsc_sum_position → Excluded because `gsc_avg_position` is the preferred ranking metric.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Verify the Grain

This checks whether one row truly represents one content item for one client on one report date.

In [5]:
grain_check = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="count")
)

duplicates = grain_check[grain_check["count"] > 1]

print("Duplicate rows:", len(duplicates))
duplicates.head()

Duplicate rows: 0


,report_date,client_hash_id,content_hash_id,count


### Query 2: Row Count and Date Span

In [6]:
print("Total Rows:", len(march_df))
print("Start Date:", march_df["report_date"].min())
print("End Date:", march_df["report_date"].max())

Total Rows: 9841378
Start Date: 2026-03-01
End Date: 2026-03-31


###Query 3:Missing values

In [7]:
missing = (
    march_df.isnull()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(missing)

gsc_avg_position            63.31
ga4_data_available          30.67
ga4_sessions                30.67
ga4_pageviews               30.67
ga4_users                   30.67
ai_chatgpt                  30.67
sessions_ai                 30.67
sessions_paid               30.67
sessions_social             30.67
sessions_referral           30.67
sessions_direct             30.67
sessions_organic            30.67
ga4_total_engagement_sec    30.67
ga4_engaged_sessions        30.67
ai_claude                   30.67
ai_perplexity               30.67
ai_gemini                   30.67
ai_copilot                  30.67
ai_other                    30.67
ai_meta                     30.67
scroll_events               30.67
gsc_clicks                   0.00
report_date                  0.00
gsc_impressions              0.00
gsc_sum_position             0.00
client_hash_id               0.00
content_hash_id              0.00
client_has_gsc               0.00
client_has_ga4               0.00
gsc_data_avail

###Query 4 — Availability Check

In [8]:
available = march_df[march_df["gsc_data_available"] == True]

print("Rows with GSC data available:", len(available))
print("Percentage:", round(len(available) / len(march_df) * 100, 2), "%")

Rows with GSC data available: 3611061
Percentage: 36.69 %


### 3. Five Features

The following five features are used for Content Refresh Opportunity Scoring.

### 1. gsc_impressions
Knowable at the decision moment because impressions are historical Search Console metrics available before deciding whether to refresh content.

### 2. gsc_clicks
Knowable at the decision moment because click data is already available from historical performance.

### 3. gsc_avg_position
Knowable at the decision moment because it reflects historical search ranking before the prediction is made.

### 4. ga4_pageviews
Knowable at the decision moment because pageview history is collected before making the refresh decision.

### 5. ga4_sessions
Knowable at the decision moment because session data represents past user engagement.

In [9]:
features = march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
    ]
].copy()

features.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,NaN,NaN
1,1,0,0.000000,NaN,NaN
2,125,1,4.928000,NaN,NaN
3,7,0,4.000000,NaN,NaN
4,11,0,2.272727,NaN,NaN


### Leakage Demonstration

To demonstrate feature leakage, I intentionally added a label-derived feature to the feature set. This caused the model to achieve an unrealistically high score because the model was given information that would not be available at prediction time.

After removing the leaked feature, the model performance became more realistic. This demonstrates why future or label-derived information must never be included as model features.

###Model without leakage

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Taking a small sample
sample = march_df.sample(5000, random_state=42)

sample["label"] = (sample["gsc_clicks"] > sample["gsc_clicks"].median()).astype(int)

X = sample[["gsc_impressions", "gsc_avg_position", "ga4_pageviews"]].fillna(0)
y = sample["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# Predict
pred = model.predict(X_test)

print("Accuracy without leakage:", accuracy_score(y_test, pred))

Accuracy without leakage: 0.942


###Adding leakage

In [11]:
# Adding the label as a feature (BAD PRACTICE)
X["leak"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy with leakage:", accuracy_score(y_test, pred))

Accuracy with leakage: 1.0


###Deleting leakage

In [12]:
# Removing the leaked feature
X = X.drop(columns=["leak"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy after removing leakage:", accuracy_score(y_test, pred))

Accuracy after removing leakage: 0.947


### Leakage Result

When I added the label as a feature, the model achieved an unrealistically high accuracy because it was given the correct answer during training. After removing the leaked feature, the accuracy became more realistic. This demonstrates why label-derived or future information should never be used as model features.

## 4. Data Limits

This dataset has the following limitations:

- Different clients have different amounts of historical Search Console (GSC) and Google Analytics 4 (GA4) data, so the available history is not balanced across all clients.
- Some rows have `gsc_data_available` or `ga4_data_available` set to **FALSE**, meaning performance metrics are unavailable for those dates and must be filtered before analysis.
- This dataset contains historical performance metrics only. It can help identify content refresh opportunities, but it cannot prove that refreshing a page will cause future performance improvements.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.